# 07 · Apply the visual-inspection flags to the master catalog

Merges the results of the three visual inspections (notebooks 04a–06a) into the merged catalog:

1. Replace `p_petromag_r` by `p_fibermag_r` for objects with an invalid Petrosian magnitude (`photflag = 1`).
2. Add `galaxyflag` (visual galaxy classification) and drop objects that are not galaxies but had a redshift (galaxy fragments).
3. Fix one MMT redshift that was attached to a fragment of its galaxy.
4. Build `extended_source_flag` from SDSS probPSF, overriding it with the visual point-source inspection.

**Inputs**
- `A2199_mastercat_intermediate_file0.csv`
- `04d_A2199galaxy_visual_classification_result.csv`, `05d_point_source_withz_SDSSimglist_result.csv`, `06d_petromag_validity_check_SDSSimglist_result.csv`

**Output**
- `A2199_mastercat_intermediate_file1_flag_update.csv`

In [1]:
# ============================================================================
# Setup
# ============================================================================
import numpy as np
import pandas as pd

# Show every column when a DataFrame is displayed
pd.set_option('display.max_columns', None)

In [2]:
# Merged catalog (photometry + redshifts from all sources)
df = pd.read_csv('./A2199_mastercat_intermediate_file0.csv')

# Galactic-extinction-corrected model and Petrosian magnitudes (suffix "_0")
for band in ['u', 'g', 'r', 'i', 'z']:
    df[f'p_modelmag_{band}_0'] = df[f'p_modelmag_{band}'] - df[f'p_extinction_{band}']
for band in ['u', 'g', 'r', 'i', 'z']:
    df[f'p_petromag_{band}_0'] = df[f'p_petromag_{band}'] - df[f'p_extinction_{band}']
df['grmod'] = df['p_modelmag_g_0'] - df['p_modelmag_r_0']

In [3]:
galaxy_check_vis = pd.read_csv('./04d_A2199galaxy_visual_classification_result.csv')      # galaxyflag
pointsource_flag_vis = pd.read_csv('./05d_point_source_withz_SDSSimglist_result.csv')       # SourceFlag
petromag_flag_vis = pd.read_csv('./06d_petromag_validity_check_SDSSimglist_result.csv')     # Flag (Petrosian validity)

## 1. Replace invalid Petrosian magnitudes by the fiber magnitude (`photflag`)

In [4]:
# Objects whose Petrosian magnitude was judged invalid
vis_phot_update_target = petromag_flag_vis[petromag_flag_vis['Flag'] == 0].copy()
vis_phot_update_target.rename(columns={'objid': 'p_objid'}, inplace=True)
target_objids = set(vis_phot_update_target['p_objid'])

update_mask = df['p_objid'].isin(target_objids)

# Use the fiber magnitude instead, and record it in photflag (1 = fiber magnitude, 0 = Petrosian)
df.loc[update_mask, 'p_petromag_r'] = df.loc[update_mask, 'p_fibermag_r']
df['photflag'] = 0
df.loc[update_mask, 'photflag'] = 1

## 2. Galaxy classification flag (`galaxyflag`)

In [5]:
# Visual galaxy classification
df = df.merge(galaxy_check_vis[['p_objid', 'galaxyflag']], on='p_objid', how='left')

# Non-galaxies that nevertheless have a redshift: galaxy fragments (checked in NED) -> removed
remove_mask = (df['galaxyflag'] == 0) & (df['z_tot_z'] != -9)
removed_objids = df.loc[remove_mask, 'p_objid']

print("Removed: sources classified as non-galaxies that nevertheless have a redshift; NED shows they are galaxy fragments.")
print(f"Number of sources removed: {remove_mask.sum()}")
print(removed_objids)

# Objects that were not inspected are not galaxies
df['galaxyflag'] = df['galaxyflag'].fillna(0).astype(int)

df = df.loc[~remove_mask].reset_index(drop=True)

Removed: sources classified as non-galaxies that nevertheless have a redshift; NED shows they are galaxy fragments.
Number of sources removed: 1
6784    1237659326566039624
Name: p_objid, dtype: int64


## 3. An MMT redshift attached to the wrong fragment

In [6]:
df[df['p_objid'] == 1237659326566236561]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag
10868,DR9,8,25.151339,1237659326566236561,247.569504,39.823536,20.86823,0.198283,20.51311,0.224491,20.63806,0.321844,20.3703,0.346669,20.57846,0.651303,20.79026,0.076691,20.39696,0.027515,20.52836,0.036063,20.34023,0.046999,20.43827,0.160604,20.31249,0.073684,19.6076,0.056651,19.40737,0.060058,19.1467,0.065085,19.17897,0.106244,0.057319,0.042175,0.030589,0.023194,0.016445,1.293596,0.094087,0.48958,0.452035,3225,301,5,238,1.230688,0,23.04,/Users/hhwang/Research/Work/MMTraw/2019.0429/r...,skysub_a2199a19_1,250.a2199a19_1_1260.ms.fits,0.070268,0.000018,N,NN,-9.0,-9.0,-9.0,-9.0,NN,NaN,-9.0,-9.0,NaN,NaN,0.070268,MMT,0.000018,20.732941,20.354785,20.497771,20.317036,20.421825,20.810911,20.470935,20.607471,20.347106,20.562015,-0.142986,0,1


In [7]:
df[df['p_objid'] == 1237659326566236559]

,phot_source,p_phtype0,p_radgal,p_objid,p_ra,p_dec,p_petromag_u,p_petromagerr_u,p_petromag_g,p_petromagerr_g,p_petromag_r,p_petromagerr_r,p_petromag_i,p_petromagerr_i,p_petromag_z,p_petromagerr_z,p_modelmag_u,p_modelmagerr_u,p_modelmag_g,p_modelmagerr_g,p_modelmag_r,p_modelmagerr_r,p_modelmag_i,p_modelmagerr_i,p_modelmag_z,p_modelmagerr_z,p_fibermag_u,p_fibermagerr_u,p_fibermag_g,p_fibermagerr_g,p_fibermag_r,p_fibermagerr_r,p_fibermag_i,p_fibermagerr_i,p_fibermag_z,p_fibermagerr_z,p_extinction_u,p_extinction_g,p_extinction_r,p_extinction_i,p_extinction_z,p_petrorad_r,p_petroraderr_r,p_devrad_i,p_devab_i,p_run,p_rerun,p_camcol,p_field,p_efac,p_probpsf,z_mmt_xcr,z_mmt_tfilename,z_dfilename,z_filename,z_mmt_z,z_mmt_zerr,z_mmt_velqual,z_ned_name,z_ned_z,z_ned_zerr,z_sdss_z,z_sdss_zerr,z_desi_id,SURVEY,z_desi_z,z_desi_zerr,FLUX_R,FLUX_IVAR_R,z_tot_z,z_tot_zsource,z_tot_zerr,p_modelmag_u_0,p_modelmag_g_0,p_modelmag_r_0,p_modelmag_i_0,p_modelmag_z_0,p_petromag_u_0,p_petromag_g_0,p_petromag_r_0,p_petromag_i_0,p_petromag_z_0,grmod,photflag,galaxyflag
10881,DR9,9,25.19616,1237659326566236559,247.570895,39.823441,17.85694,0.038351,16.79143,0.035355,16.4286,0.081272,16.29547,0.291583,16.32974,0.477721,17.97785,0.022511,16.78998,0.004393,16.44835,0.004312,16.23527,0.005545,16.18529,0.015291,19.90454,0.033805,18.68609,0.024421,18.25266,0.051256,17.99365,0.163092,17.84956,0.18551,0.057412,0.042243,0.030638,0.023232,0.016472,6.345457,0.149922,6.62471,0.740933,3225,301,5,238,-1.824057,0,-9.0,NN,NN,nn.fits,-9.0,-9.0,N,2MASS J16301697+3949236,0.0701,0.000007,0.0701,0.000007,NN,NaN,-9.0,-9.0,NaN,NaN,0.0701,SDSS,0.000007,17.920438,16.747737,16.417712,16.212038,16.168818,17.799528,16.749187,16.397962,16.272238,16.313268,0.330025,0,1


1237659326566236559 and 1237659326566236561 appear as two objects but are fragments of the same galaxy (1237659326566236559 is the centre).

1237659326566236559 carries an SDSS redshift and 1237659326566236561 an MMT redshift; the two values are nearly identical, so the MMT redshift belongs to 1237659326566236559.

The MMT columns are therefore copied from 1237659326566236561 to 1237659326566236559, and the duplicate row 1237659326566236561 is removed.

In [8]:
cols = [
    'z_mmt_xcr', 'z_mmt_tfilename', 'z_dfilename', 'z_filename',
    'z_mmt_z', 'z_mmt_zerr', 'z_mmt_velqual',
    'z_tot_z', 'z_tot_zsource', 'z_tot_zerr'
]

src_id = 1237659326566236561   # row to copy from (fragment)
dst_id = 1237659326566236559   # row to update (galaxy centre)

src_vals = df.loc[df['p_objid'] == src_id, cols].iloc[0]
df.loc[df['p_objid'] == dst_id, cols] = src_vals.values

df = df[df['p_objid'] != src_id].copy()

## 4. Extended-source flag

In [9]:
# Point sources with a redshift judged invalid by the inspection (removed below)
pointsource_flag_vis[pointsource_flag_vis['SourceFlag'] == 0]

,objid,RA,DEC,SourceFlag
141,1237659326566039619,247.163557,40.125913,0
158,1237659326566039629,247.164539,40.120931,0
213,1237659326566236252,247.641373,39.830893,0


In [10]:
# Point sources with a redshift judged extended by the inspection
pointsource_flag_vis[pointsource_flag_vis['SourceFlag'] == 2]

,objid,RA,DEC,SourceFlag
66,1237659330315288687,246.929210,39.381253,2
85,1237655471820898598,247.550113,39.553527,2


In [11]:
# Attach the point-source inspection result
point_flags = pointsource_flag_vis[['objid', 'SourceFlag']].rename(columns={'objid': 'p_objid'})
df = df.merge(point_flags, on='p_objid', how='left')

# Invalid sources are removed
remove_mask = df['SourceFlag'] == 0
print(f"Removing rows with SourceFlag == 0: {remove_mask.sum()}")
df = df.loc[~remove_mask].copy()

# Objects that were not inspected get -9
df.loc[df['SourceFlag'].isna(), 'SourceFlag'] = -9

df.reset_index(drop=True, inplace=True)

Removing rows with SourceFlag == 0: 3


`extended_source_flag`: 1 = extended, 0 = point source. Base value from SDSS probPSF, overridden to 1 where the inspection found an extended source.

In [12]:
# 1) probPSF == 1 -> point source (0); probPSF == 0 -> extended (1)
df["extended_source_flag"] = df["p_probpsf"].map({1: 0, 0: 1})

# 2) Visual override: SourceFlag == 2 -> extended
if "SourceFlag" in df.columns:
    override_mask = df["SourceFlag"] == 2
    df.loc[override_mask, "extended_source_flag"] = 1
    print(f"SourceFlag==2 override count: {override_mask.sum()}")

    # 3) The temporary column is no longer needed
    df.drop(columns=["SourceFlag"], inplace=True)
    print("Dropped column: SourceFlag")
else:
    print("SourceFlag column not found; only p_probpsf mapping applied.")

print(df[["p_objid", "p_probpsf", "extended_source_flag"]].head())

SourceFlag==2 override count: 2
Dropped column: SourceFlag
               p_objid  p_probpsf  extended_source_flag
0  1237659325492298748          0                     1
1  1237659325492298738          1                     0
2  1237659325492298685          0                     1
3  1237659325492298214          0                     1
4  1237659325492298822          1                     0


In [13]:
df.to_csv('A2199_mastercat_intermediate_file1_flag_update.csv', index=False)